In [22]:
import pandas as pd
import geopandas as gpd
from pathlib import Path

Adding geo-data to df

In [23]:
# --- 1. Load Your Data ---

# Path to the CSV file we are enriching
csv_path = Path('../clean/doctors_clean.csv')
df = pd.read_csv(csv_path)

# Path to the GeoJSON file (in the same 'scripts' folder)
geojson_path = 'lor_ortsteile.geojson'
gdf_polygons = gpd.read_file(geojson_path)


# --- 2. Create GeoDataFrames ---

# Convert your DataFrame of post offices into a GeoDataFrame
gdf_points = gpd.GeoDataFrame(
    df, geometry=gpd.points_from_xy(df.longitude, df.latitude), crs="EPSG:4326"
)

# Ensure both GeoDataFrames use the same Coordinate Reference System (CRS)
gdf_polygons = gdf_polygons.to_crs(gdf_points.crs)


# --- 3. Perform the Spatial Join ---

# This finds which polygon (neighborhood) each point is in
gdf_joined = gpd.sjoin(gdf_points, gdf_polygons, how="left", predicate='within')


# --- 4. Add the New Columns to Your Original DataFrame ---

# We use the final column names we identified
district_col_name = 'BEZIRK'
neighborhood_col_name = 'OTEIL'

# Add the new columns from the joined data back to your original DataFrame
df['district'] = gdf_joined[district_col_name].reset_index(drop=True)
df['neighborhood'] = gdf_joined[neighborhood_col_name].reset_index(drop=True)


# --- 5. Check the Result ---
print("New columns have been added successfully! ✅")
# The redundant 'neighborhood_id' column has been removed from the check
print(df[['city', 'district', 'neighborhood']].head())

New columns have been added successfully! ✅
     city             district neighborhood
0  Berlin  Marzahn-Hellersdorf    Kaulsdorf
1  Berlin                Mitte      Wedding
2  Berlin              Spandau      Staaken
3  Berlin  Marzahn-Hellersdorf    Kaulsdorf
4  Berlin  Marzahn-Hellersdorf     Biesdorf


In [24]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1643 entries, 0 to 1642
Data columns (total 21 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   city                    1512 non-null   object 
 1   housenumber             1335 non-null   object 
 2   postcode                1571 non-null   float64
 3   street                  1639 non-null   object 
 4   amenity                 1643 non-null   object 
 5   speciality              1433 non-null   object 
 6   name                    1611 non-null   object 
 7   opening_hours           1130 non-null   object 
 8   website                 910 non-null    object 
 9   id                      1643 non-null   int64  
 10  longitude               1643 non-null   float64
 11  latitude                1643 non-null   float64
 12  country                 755 non-null    object 
 13  suburb                  1105 non-null   object 
 14  wheelchair              582 non-null    

Now I need to check that all doctors are within Berlin.

In [25]:
from shapely.ops import unary_union
import geopandas as gpd # Import in case this is in a new cell

# --- 1. Create a single "Berlin" polygon ---
# We use 'gdf_polygons', which you already loaded and projected
berlin_boundary = gpd.GeoSeries(unary_union(gdf_polygons.geometry), crs=gdf_polygons.crs)

# --- 2. Perform the check ---
# We use 'gdf_points', which you already created
# .within() checks if each point is inside the berlin_boundary
is_inside_berlin = gdf_points.within(berlin_boundary.geometry[0])

# --- 3. Report the results ---
num_outside = (~is_inside_berlin).sum() # ~ inverts True/False, counting the Falses (those outside)

if num_outside == 0:
    print("✅ All points are correctly located within the Berlin boundaries.")
else:
    print(f"⚠️ Warning: Found {num_outside} point(s) outside the Berlin boundaries.")
    
    # (Optional) Show the rows that are outside
    # We use the original 'df' for a clean text report
    print("\nClubs located outside Berlin:")
    print(df[~is_inside_berlin][['club_name', 'street', 'city', 'latitude', 'longitude']])

✅ All points are correctly located within the Berlin boundaries.


As all doctors are within Berlin, I'll fill missing values in the 'city' column and drop 'country' column.

In [26]:
df['city'].unique()

array(['Berlin', nan], dtype=object)

In [27]:
df['city'] = df['city'].fillna('Berlin')
df.drop(columns='country', inplace=True)

In [28]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1643 entries, 0 to 1642
Data columns (total 20 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   city                    1643 non-null   object 
 1   housenumber             1335 non-null   object 
 2   postcode                1571 non-null   float64
 3   street                  1639 non-null   object 
 4   amenity                 1643 non-null   object 
 5   speciality              1433 non-null   object 
 6   name                    1611 non-null   object 
 7   opening_hours           1130 non-null   object 
 8   website                 910 non-null    object 
 9   id                      1643 non-null   int64  
 10  longitude               1643 non-null   float64
 11  latitude                1643 non-null   float64
 12  suburb                  1105 non-null   object 
 13  wheelchair              582 non-null    object 
 14  description             104 non-null    

Now we need to add a column district_id and neighborhood_id from database tables districts.csv and neighborhoods.csv 

In [29]:
# --- Load the new lookup tables ---

districts_path = Path('../source/districts.csv')
neighborhoods_path = Path('../source/neighborhoods.csv')

# Read the CSV files into DataFrames
districts_df = pd.read_csv(districts_path)
neighborhoods_df = pd.read_csv(neighborhoods_path)

# --- Inspect the DataFrames ---

print("--- Districts Table Info ---")
districts_df.info()
print("\n--- Districts Table Head ---")
print(districts_df.head())

print("\n" + "="*50 + "\n") # A separator for clarity

print("--- Neighborhoods Table Info ---")
neighborhoods_df.info()
print("\n--- Neighborhoods Table Head ---")
print(neighborhoods_df.head())

--- Districts Table Info ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12 entries, 0 to 11
Data columns (total 3 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   district_id  12 non-null     int64 
 1   district     12 non-null     object
 2   geometry     12 non-null     object
dtypes: int64(1), object(2)
memory usage: 420.0+ bytes

--- Districts Table Head ---
   district_id                    district  \
0     11012012               Reinickendorf   
1     11004004  Charlottenburg-Wilmersdorf   
2     11009009            Treptow-Köpenick   
3     11003003                      Pankow   
4     11008008                    Neukölln   

                                            geometry  
0  MULTIPOLYGON (((13.320744327762688 52.62659906...  
1  MULTIPOLYGON (((13.321109641281137 52.52446299...  
2  MULTIPOLYGON (((13.579253945950567 52.39083025...  
3  MULTIPOLYGON (((13.504807966473637 52.61959821...  
4  MULTIPOLYGON (((13.45832

In [30]:
# Merge with Districts Table to add 'district_id'

# We only need the ID and the key from the districts table
districts_lookup = districts_df[['district_id', 'district']]

# Perform the merge
# 'how="left"' keeps all rows from original 'df'
df = pd.merge(df, districts_lookup, on='district', how='left')


# Merge with Neighborhoods Table to add 'neighborhood_id'

# We only need the ID and the key from the neighborhoods table
neighborhoods_lookup = neighborhoods_df[['neighborhood_id', 'neighborhood']]

# Perform the second merge
df = pd.merge(df, neighborhoods_lookup, on='neighborhood', how='left')

print("IDs have been added successfully! ✅")
print(df[['district', 'district_id', 'neighborhood', 'neighborhood_id', 'suburb']].head())

IDs have been added successfully! ✅
              district  district_id neighborhood  neighborhood_id     suburb
0  Marzahn-Hellersdorf     11010010    Kaulsdorf             1003        NaN
1                Mitte     11001001      Wedding              105    Wedding
2              Spandau     11005005      Staaken              504        NaN
3  Marzahn-Hellersdorf     11010010    Kaulsdorf             1003  Kaulsdorf
4  Marzahn-Hellersdorf     11010010     Biesdorf             1002   Biesdorf


Next, I'm removing the 'district', 'neighborhood', and 'suburb' columns, as they are superfluous.

In [31]:
df.drop(columns=['district', 'neighborhood', 'suburb'], inplace=True, errors='ignore')
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1643 entries, 0 to 1642
Data columns (total 19 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   city                    1643 non-null   object 
 1   housenumber             1335 non-null   object 
 2   postcode                1571 non-null   float64
 3   street                  1639 non-null   object 
 4   amenity                 1643 non-null   object 
 5   speciality              1433 non-null   object 
 6   name                    1611 non-null   object 
 7   opening_hours           1130 non-null   object 
 8   website                 910 non-null    object 
 9   id                      1643 non-null   int64  
 10  longitude               1643 non-null   float64
 11  latitude                1643 non-null   float64
 12  wheelchair              582 non-null    object 
 13  description             104 non-null    object 
 14  email                   189 non-null    

We need to change some columns types.

In [32]:
# This handles NaNs and removes the .0 from floats
df['postcode'] = df['postcode'].astype(pd.Int64Dtype()).astype(str).replace('<NA>', None)

# Convert IDs to string
df['id'] = df['id'].astype(str)
df['district_id'] = df['district_id'].astype(str)
df['neighborhood_id'] = df['neighborhood_id'].astype(str)

df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1643 entries, 0 to 1642
Data columns (total 19 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   city                    1643 non-null   object 
 1   housenumber             1335 non-null   object 
 2   postcode                1571 non-null   object 
 3   street                  1639 non-null   object 
 4   amenity                 1643 non-null   object 
 5   speciality              1433 non-null   object 
 6   name                    1611 non-null   object 
 7   opening_hours           1130 non-null   object 
 8   website                 910 non-null    object 
 9   id                      1643 non-null   object 
 10  longitude               1643 non-null   float64
 11  latitude                1643 non-null   float64
 12  wheelchair              582 non-null    object 
 13  description             104 non-null    object 
 14  email                   189 non-null    

In [33]:
# Find the index of the row(s) where 'name' is null 
index_to_drop = df[df['name'].isnull()].index
print(index_to_drop)

Index([   3,   51,   76,  140,  387,  388,  389,  463,  518,  558,  629,  693,
        708,  871,  883,  937, 1021, 1185, 1186, 1244, 1268, 1319, 1320, 1321,
       1322, 1344, 1406, 1543, 1608, 1628, 1630, 1631],
      dtype='int64')


I would like to inspect the rows where the 'name' column is not filled, and fill them if possible.

In [34]:
# Create a mask for the rows where 'name' is null
is_name_missing = df['name'].isnull()

# The columns to see
columns_to_show = ['name', 'website','speciality']

# Print the result (all rows, but only the 3 columns)
print(f"--- {is_name_missing.sum()} rows where 'name' is missing ---")
print("--- Showing only 'name' and 'website' ---")

# .to_string() forces pandas to print ALL the rows,
# not hide them behind "..."
print(df[is_name_missing][columns_to_show].to_string())

--- 32 rows where 'name' is missing ---
--- Showing only 'name' and 'website' ---
     name                                                                     website                          speciality
3     NaN                                         https://www.drhenriettefriedrich.de                                 NaN
51    NaN                                                                         NaN                         gynaecology
76    NaN                                          https://www.berliner-augenarzt.de/                       ophthalmology
140   NaN                                                                         NaN                                 NaN
387   NaN                                            https://www.mvz-berlin-rudow.de/                            internal
388   NaN                                            https://www.mvz-berlin-rudow.de/                        orthopaedics
389   NaN                                            https://www

I'll fill names from websites manuly where possible.

In [35]:
# Create a filter for the rows we need to fix
# (where 'name' IS null AND 'website' IS NOT null)
condition_to_fix = df['name'].isnull() & df['website'].notna()

# Create a small DataFrame with just these rows
df_to_fix = df[condition_to_fix]

# Define columns to show
# I added 'amenity' and 'speciality' as context
columns_to_show = ['name', 'website', 'amenity', 'speciality']
existing_cols = [col for col in columns_to_show if col in df.columns]

print(f"--- Found {len(df_to_fix)} rows to fix manually ---")
print("These have a 'website' but no 'name':\n")

# Print the list of websites to check
print(df_to_fix[existing_cols].to_string())

--- Found 13 rows to fix manually ---
These have a 'website' but no 'name':

     name                                                                     website  amenity                          speciality
3     NaN                                         https://www.drhenriettefriedrich.de  doctors                                 NaN
76    NaN                                          https://www.berliner-augenarzt.de/  doctors                       ophthalmology
387   NaN                                            https://www.mvz-berlin-rudow.de/  doctors                            internal
388   NaN                                            https://www.mvz-berlin-rudow.de/  doctors                        orthopaedics
389   NaN                                            https://www.mvz-berlin-rudow.de/  doctors                             general
518   NaN                                      https://www.gastroenterologie-horn.de/  doctors           internal;gastroenterology
693   

In [36]:
# We use .loc[index, column_name] = value

df.loc[3, 'name'] = 'Dr. Henriette Friedrich'
df.loc[3, 'speciality'] = 'general' # 'general' is a common OSM tag for this

df.loc[76, 'name'] = 'Christoph J. Huber'

df.loc[387, 'name'] = 'MVZ Berlin Rudow'
df.loc[388, 'name'] = 'MVZ Berlin Rudow'
df.loc[389, 'name'] = 'MVZ Berlin Rudow'

df.loc[518, 'name'] = 'Dr. med. Andreas Horn'

df.loc[693, 'name'] = 'Arztpraxis Berlin Friedenau'

df.loc[708, 'name'] = 'Dr. Strunz'

df.loc[1021, 'name'] = 'Kinderarzt Zimmermann'

df.loc[1244, 'name'] = 'Dr. med. Thorsten Löbbert'

df.loc[1321, 'name'] = 'Dr. Nicolai Sedlaczek, Dr. Moritz Knies, Prof. Dr. Saskia Rohrbach, Dr. Tobias Maier'

df.loc[1406, 'name'] = 'Dr. med. Kathrin Irrgang'

df.loc[1608, 'name'] = 'Medicum'
df.loc[1608, 'speciality'] = 'surgery, dermatology, gynaecology, ear_nose_throat, internal, cardiology, dentistry'

In [37]:
# Create a filter for the rows we need to check
# (where 'name' IS null AND 'website' IS ALSO null)
condition_to_check = df['name'].isnull() & df['website'].isnull()

# Create a small DataFrame with just these rows
df_to_check = df[condition_to_check]

# Define columns to show
# We now want to see the address and context columns
columns_to_show = [
    'name',        # (This column will be all NaN)
    'website',     # (This column will be all NaN)
    'street',      
    'housenumber', 
    'postcode',    
    'amenity',     
    'speciality'   # (Context: Does it have a speciality?)
]
existing_cols = [col for col in columns_to_show if col in df.columns]

print(f"--- Found {len(df_to_check)} rows with NO 'name' AND NO 'website' ---")
print("Checking if they have address or speciality data:\n")

# .to_string() ensures all rows are printed
print(df_to_check[existing_cols].to_string())

--- Found 19 rows with NO 'name' AND NO 'website' ---
Checking if they have address or speciality data:

     name website                  street housenumber postcode  amenity            speciality
51    NaN     NaN          Skarbinastraße          79    12309  doctors           gynaecology
140   NaN     NaN          Berliner Allee          97    13088  doctors                   NaN
463   NaN     NaN             Kormoranweg          31     None  doctors          radiotherapy
558   NaN     NaN    Wilmersdorfer Straße          45     None  doctors         ophthalmology
629   NaN     NaN           Dunckerstraße         70A    10437  doctors      child_psychiatry
871   NaN     NaN    Martin-Luther-Straße         124    10825  doctors       plastic_surgery
883   NaN     NaN     Wilhelm-Blos-Straße          61    12623  doctors               general
937   NaN     NaN     Hugo-Distler-Straße          24    12619  doctors   general;paediatrics
1185  NaN     NaN             Hauptstraße        

These rows are being dropped, as I was unable to source the required information for them.

Additionally, I've observed that the 'amenity' column is unreliable due to inconsistent data entry. Some clinics have a separate record for each specialty, while others list all specialties in a single 'doctors' record. I want to fix it.

In [38]:
df['name'].value_counts()

name
Ärztehaus                               25
Praxis für Allgemeinmedizin              6
MVZ Berlin Rudow                         4
Praxis für Kinder- und Jugendmedizin     4
Hausarztpraxis                           4
                                        ..
Ärztehaus Nordflügel                     1
Ärztehaus Südflügel                      1
Helios Arthropädicum Kaulsdorf           1
Medical Park Berlin Humboldtmühle        1
Kinderarztpraxis am Schlachtensee        1
Name: count, Length: 1560, dtype: int64

In [40]:
print(f"Original size before aggregation: {len(df)} rows")

# Custom Aggregation Function
def aggregate_row(rows):
    
    # Take the *first* value for most columns
    data = rows.iloc[0].to_dict()
    
    # Get all unique, non-empty specialities from all duplicate rows
    all_specialities = rows['speciality'].dropna().unique()
    
    # Join them with a comma
    data['speciality'] = ', '.join(all_specialities)
      
    # We ONLY set amenity to 'clinic' IF we ACTUALLY found more than one speciality for this *exact* object.
    if len(all_specialities) > 1:
        data['amenity'] = 'clinic'
       
    return pd.Series(data)

# Group by 'name' AND 'ADDRESS' and apply our function
df = df.groupby(
    ['name', 'street', 'housenumber'], # Group by unique object
    as_index=False,
    dropna=False # Keep rows where name/street might be NaN
).apply(aggregate_row, include_groups=False) # 'include_groups=False' silences the warning

print(f"New size after aggregation: {len(df)} rows")

# Check 'MVZ Berlin Rudow' example
# This should now be 1 row, with 'amenity=clinic' and a combined 'speciality' list.
print("\n Checking 'MVZ Berlin Rudow' ")
print(df[df['name'] == 'MVZ Berlin Rudow'][['name', 'amenity', 'speciality', 'street']])

# Check 'Ärztehaus' example
# This should still be 25 rows, and their 'amenity' will *not* be changed (unless one of them # had duplicates at the *same address*). This is correct.
print("\n Checking 'Ärztehaus' ")
print(f"Total 'Ärztehaus' entries: {len(df[df['name'] == 'Ärztehaus'])}")

Original size before aggregation: 1643 rows
New size after aggregation: 1636 rows

 Checking 'MVZ Berlin Rudow' 
                  name amenity                       speciality  \
1001  MVZ Berlin Rudow  clinic  internal, orthopaedics, general   

                       street  
1001  Waßmannsdorfer Chaussee  

 Checking 'Ärztehaus' 
Total 'Ärztehaus' entries: 25


In [42]:
# Get a count of the rows we are about to drop
# These are the "ghost rows" that we couldn't save (where 'name' is still NaN).
ghost_rows_count = df['name'].isnull().sum()

print(f"Found {ghost_rows_count} rows that still have no 'name'.")

# Drop all rows where 'name' is still NaN
df.dropna(subset=['name'], inplace=True)

print(f"Final clean size: {len(df)} rows.")

Found 0 rows that still have no 'name'.
Final clean size: 1619 rows.


In [43]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1619 entries, 0 to 1618
Data columns (total 19 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   name                    1619 non-null   object 
 1   street                  1615 non-null   object 
 2   housenumber             1311 non-null   object 
 3   city                    1619 non-null   object 
 4   postcode                1549 non-null   object 
 5   amenity                 1619 non-null   object 
 6   speciality              1619 non-null   object 
 7   opening_hours           1118 non-null   object 
 8   website                 906 non-null    object 
 9   id                      1619 non-null   object 
 10  longitude               1619 non-null   float64
 11  latitude                1619 non-null   float64
 12  wheelchair              576 non-null    object 
 13  description             102 non-null    object 
 14  email                   188 non-null    objec

In [44]:
#Save the Final Enriched File
save_path = Path('../clean/doctors_clean_with_distr.csv')
df.to_csv(save_path, index=False, encoding='utf-8-sig')
print(f"\nDataFrame successfully saved to '{save_path}'")


DataFrame successfully saved to '..\clean\doctors_clean_with_distr.csv'


Validation & Quality Checks:
  - Check for duplicate rows. 
  - Check final row count.

In [45]:
df.duplicated().sum()

np.int64(0)

In [46]:
df['district_id'].isnull().sum()

np.int64(0)

In [47]:
df['neighborhood_id'].isnull().sum()

np.int64(0)

In [48]:
df['id'].nunique()

1619

In [49]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1619 entries, 0 to 1618
Data columns (total 19 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   name                    1619 non-null   object 
 1   street                  1615 non-null   object 
 2   housenumber             1311 non-null   object 
 3   city                    1619 non-null   object 
 4   postcode                1549 non-null   object 
 5   amenity                 1619 non-null   object 
 6   speciality              1619 non-null   object 
 7   opening_hours           1118 non-null   object 
 8   website                 906 non-null    object 
 9   id                      1619 non-null   object 
 10  longitude               1619 non-null   float64
 11  latitude                1619 non-null   float64
 12  wheelchair              576 non-null    object 
 13  description             102 non-null    object 
 14  email                   188 non-null    objec